In [2]:
# MACHINE LEARNING ASSIGNMENT - CAR PRICE PREDICTION

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import warnings
warnings.filterwarnings('ignore')


# 2. LOAD DATASET

df = pd.read_csv('CarPrice_Assignment.csv')

print("Dataset Loaded Successfully!\n")
print(df.head())


# 3. DATA PREPROCESSING

# Drop unnecessary column
if 'car_ID' in df.columns:
    df.drop(['car_ID'], axis=1, inplace=True)

# Extract Car Brand
df['CarBrand'] = df['CarName'].apply(lambda x: x.split(' ')[0])
df.drop(['CarName'], axis=1, inplace=True)

# Encode categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

print("\nMissing Values:\n", df.isnull().sum())


# 4. DEFINE FEATURES & TARGET

X = df.drop('price', axis=1)
y = df['price']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# 5. MODEL IMPLEMENTATION

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Support Vector Regressor": SVR()
}

results = {}

print("\n===== MODEL PERFORMANCE =====\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    results[name] = {"R2": r2, "MSE": mse, "MAE": mae}

    print(f"{name}:")
    print(f"R2 Score: {r2}")
    print(f"MSE: {mse}")
    print(f"MAE: {mae}")
    print("---------------------------")


# 6. BEST MODEL SELECTION + JUSTIFICATION

best_model_name = max(results, key=lambda x: results[x]["R2"])
best_metrics = results[best_model_name]

print("\n===== BEST MODEL =====\n")
print("Best Model:", best_model_name)

print("\n===== JUSTIFICATION =====\n")

for model in results:
    print(f"{model} -> R2: {results[model]['R2']}, MSE: {results[model]['MSE']}, MAE: {results[model]['MAE']}")

print(f"""
The best model is '{best_model_name}' because:
- It has the highest R2 score ({best_metrics['R2']}), meaning it explains maximum variance.
- It has lower MSE ({best_metrics['MSE']}) → fewer large errors.
- It has lower MAE ({best_metrics['MAE']}) → better average prediction accuracy.
Thus, it provides the most reliable predictions.
""")


# 7. FEATURE IMPORTANCE + SELECTION


print("\n===== FEATURE IMPORTANCE =====\n")

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
feature_names = X.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

# Select Top Features
top_n = 8
selected_features = feature_importance_df.head(top_n)['Feature'].tolist()

print("\n===== SELECTED FEATURES =====\n")
for f in selected_features:
    print(f)

print(f"""
Reason for selecting top {top_n} features:
- These features have the highest importance scores.
- They contribute the most to predicting car price.
- Removing low-importance features reduces noise and improves efficiency.
""")


# 8. MODEL USING SELECTED FEATURES

X_selected = df[selected_features]
X_selected_scaled = scaler.fit_transform(X_selected)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_selected_scaled, y, test_size=0.2, random_state=42
)

rf_selected = RandomForestRegressor(random_state=42)
rf_selected.fit(X_train_s, y_train_s)

y_pred_s = rf_selected.predict(X_test_s)

print("\n===== PERFORMANCE WITH SELECTED FEATURES =====\n")
print("R2:", r2_score(y_test_s, y_pred_s))
print("MSE:", mean_squared_error(y_test_s, y_pred_s))
print("MAE:", mean_absolute_error(y_test_s, y_pred_s))


# 9. HYPERPARAMETER TUNING

print("\n===== HYPERPARAMETER TUNING =====\n")

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

print("\nTuned Model Performance:")
print("R2:", r2_score(y_test, y_pred_tuned))
print("MSE:", mean_squared_error(y_test, y_pred_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_tuned))

print("""
Hyperparameter tuning improves model performance by selecting optimal parameter values.
""")


# 10. FINAL CONCLUSION


print("\n===== FINAL CONCLUSION =====\n")

print(f"""
- Best Model: {best_model_name}
- Selected based on highest R2 and lowest errors.
- Important features identified and used for better efficiency.
- Hyperparameter tuning improved performance further.

This model can effectively predict car prices and help in business decision making.
""")


Dataset Loaded Successfully!

   car_ID  symboling                   CarName fueltype aspiration doornumber  \
0       1          3        alfa-romero giulia      gas        std        two   
1       2          3       alfa-romero stelvio      gas        std        two   
2       3          1  alfa-romero Quadrifoglio      gas        std        two   
3       4          2               audi 100 ls      gas        std       four   
4       5          2                audi 100ls      gas        std       four   

       carbody drivewheel enginelocation  wheelbase  ...  enginesize  \
0  convertible        rwd          front       88.6  ...         130   
1  convertible        rwd          front       88.6  ...         130   
2    hatchback        rwd          front       94.5  ...         152   
3        sedan        fwd          front       99.8  ...         109   
4        sedan        4wd          front       99.4  ...         136   

   fuelsystem  boreratio  stroke compressionratio 